# 02. Spark & Delta Lake Integration Test

Notebook này dùng để kiểm tra cấu hình Spark, khả năng đọc ghi vào Delta Lake trên MinIO (S3) và tính năng Time Travel.

In [ ]:
import sys
import os

# Thêm thư mục gốc vào path để import src
sys.path.append(os.path.abspath('..'))

from src.utils.spark_session import get_spark_session
from pyspark.sql import functions as F

## 1. Khởi tạo Spark Session

In [ ]:
spark = get_spark_session("Notebook-Infrastructure-Test")
print(f"Spark Version: {spark.version}")

## 2. Kiểm tra đọc dữ liệu Local và ghi vào Delta Lake (MinIO)

In [ ]:
# Đọc thử 1000 dòng ratings
df = spark.read.csv("../dataset/ml-25m/ratings.csv", header=True, inferSchema=True).limit(1000)

# Ghi vào Delta Lake trên MinIO
TEST_PATH = "s3a://movie-data/test/delta_test_table"
df.write.format("delta").mode("overwrite").save(TEST_PATH)

print("Ghi dữ liệu vào Delta Lake thành công!")

## 3. Kiểm tra tính năng Time Travel

In [ ]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, TEST_PATH)

# Xem lịch sử phiên bản
delta_table.history().select("version", "timestamp", "operation", "operationParameters").show()

## 4. Dọn dẹp

In [ ]:
spark.stop()